In [ ]:
%reload_ext autoreload
%autoreload 2

# %% 0. 导入必要的库
import sys
import uuid
import numpy as np

sys.path.append('/home/morisi/Workspace/3D-Ocean')

from src.analysis.season import SeasonalityAnalysis
from src.analysis.rmse import plot_metrics

from src.trainer.base import BaseTrainer

from src.models.SST.ConvLSTM import ConvLSTM
from src.models.SST.UNetLSTM import UNetLSTM
from src.models.SST.Transformer import SSTTransformer
from src.models.SST.RGTransformer import RGTransformer

from src.config.area import Area
from src.config.params import MODEL_SAVE_PATH
from src.dataset.ERA5 import ERA5SSTMonthlyDataset
from src.dataset.OISST import OISSTMonthlyDataset

In [ ]:
# %% 1. 数据集分析

## 1.1 ERA5 月平均数据

### 1.1.1 时间序列的周期性分析

dataset = OISSTMonthlyDataset(
    seq_len=1,
    offset=0,
    resolution=1,
    lon=np.array([-180, 180]),  # 全球区域
    lat=np.array([-80, 80])    # 全球区域
)

analyzer = SeasonalityAnalysis(dataset)

# 绘制季节性图表
fig = analyzer.plot_seasonal_patterns()

In [ ]:
# %% 2. 模型构建

## 2.1 不同的训练器

area = Area('Global', lon=[-180, 180], lat=[-80, 80], description='全球区域')

trainer_uid = str(uuid.uuid4())
resolution = 1
seq_len = 2
offset = 0

use_checkpoint = True

width = int(area.width / resolution)
height = int(area.height / resolution)

model_path = f"{MODEL_SAVE_PATH}/seq_len-{seq_len}"

dataset_params = {
    "seq_len": seq_len,
    "offset": offset,
    "resolution": resolution,
}

trainer_params = {
    "epochs": 400,
    "batch_size": 32,
    "num_workers": 12,
    "use_wandb": False,
    "use_checkpoint": use_checkpoint
}

conv_m_params = {
    'input_dim': 1,  # 输入通道数
    'hidden_dim': 1,  # 隐藏层通道数
    'kernel_size': (5, 5),  # 卷积核大小
    'num_layers': 1,  # LSTM层数
    'bias': False  # 是否使用偏置
}

unet_lstm_m_params = {
    "input_channels": 1,
    "output_channels": 1,
    "features": [32, 64, 128, 256],
    "lstm_hidden_channels": 1024,
    "learning_rate": 1e-3
}

transformer_m_params = {
    "width": width,
    "height": height,
    "seq_len": seq_len,
    "d_model": 512,
    "nhead": 8,
    "num_encoder_layers": 2,
    "num_decoder_layers": 2,
    "dim_feedforward": 512,
    "dropout": 0.2,
    "learning_rate": 1e-3
}

conv_trainer = BaseTrainer(
    title='ConvLSTM',
    area=area,
    uid=trainer_uid,
    model_class=ConvLSTM,
    dataset_class=ERA5SSTMonthlyDataset,
    save_path=f'{model_path}/conv.pkl',
    use_checkpoint=use_checkpoint,
    dataset_params=dataset_params,
    trainer_params=trainer_params,
    model_params=conv_m_params,
)

unet_lstm_trainer = BaseTrainer(
    title='UNetLSTM',
    area=area,
    uid=trainer_uid,
    model_class=UNetLSTM,
    dataset_class=ERA5SSTMonthlyDataset,
    save_path=f'{model_path}/unet.pkl',
    use_checkpoint=use_checkpoint,
    dataset_params=dataset_params,
    trainer_params=trainer_params,
    model_params=unet_lstm_m_params,
)

transformer_trainer = BaseTrainer(
    title='Transformer',
    area=area,
    uid=trainer_uid,
    model_class=SSTTransformer,
    dataset_class=ERA5SSTMonthlyDataset,
    save_path=f'{model_path}/transformer.pkl',
    use_checkpoint=use_checkpoint,
    dataset_params=dataset_params,
    trainer_params=trainer_params,
    model_params=transformer_m_params,
)


In [ ]:

rg_transformer_m_params = {
    "width": width,
    "height": height,
    "resolution": resolution,
    "lat_range": area.lat,
    "lon_range": area.lon,
    "seq_len": seq_len,
    "d_model": 1024, 
    "num_heads": 8,
    "dim_feedforward": 1024,
    "dropout": 0.1,
    "recursion_depth": 2,
    "learning_rate": 1e-4,
}

rg_transformer_trainer = BaseTrainer(
    title='RGTransformer',
    area=area,
    uid=trainer_uid,
    model_class=RGTransformer,
    dataset_class=OISSTMonthlyDataset,
    save_path=f'{model_path}/rg_transformer.pkl',
    use_checkpoint=use_checkpoint,
    dataset_params=dataset_params,
    trainer_params=trainer_params,
    model_params=rg_transformer_m_params,
)

In [ ]:
trainers = [
    # conv_trainer,
    # unet_lstm_trainer,
    # transformer_trainer,
    rg_transformer_trainer
]

for trainer in trainers: 
    trainer.train()

In [ ]:
# %% 4. 模型评估

## 4.1 预测
_trainer = BaseTrainer(
    title='RGTransformer',
    area=area,
    uid=trainer_uid,
    model_class=RGTransformer,
    dataset_class=OISSTMonthlyDataset,
    save_path=f'{model_path}/rg_transformer.pkl',
    use_checkpoint=use_checkpoint,
    checkpoint_path=f'out/checkpoints/RGTransformer/last.ckpt',
    dataset_params=dataset_params,
    trainer_params=trainer_params,
    model_params=rg_transformer_m_params,
)


months = [ i for i in range(521, 522) ]
 
RMSE = {
    'ConvLSTM': [],
    'UNetLSTM': [],
    'Transformer': [],
    'RATransformer': []
}
R2 = {
    'ConvLSTM': [],
    'UNetLSTM': [],
    'Transformer': [],
    'RATransformer': []
}
SSTA = {
    'ConvLSTM': [],
    'UNetLSTM': [],
    'Transformer': [],
    'RATransformer': []
}

p_trainers = [
    # conv_trainer,
    # unet_lstm_trainer,
    # transformer_trainer,
    rg_transformer_trainer
]

for trainer in p_trainers:
    title = trainer.title
    
    for month in months:
        offset = month
        input, output, pred_output, rmse, r2, ssta = trainer.predict(offset, plot=True)

        RMSE[title].append(rmse)
        R2[title].append(r2)
        SSTA[title].append(ssta)

print(RMSE)
print(R2)

In [ ]:
# %% 5. 微调训练
rg_transformer_m_params = {
    "width": width,
    "height": height,
    "resolution": resolution,
    "lat_range": area.lat,
    "lon_range": area.lon,
    "seq_len": seq_len,
    "d_model": 1024, 
    "num_heads": 8,
    "dim_feedforward": 1024,
    "dropout": 0.1,
    "recursion_depth": 2,
    "learning_rate": 1.5e-4,
}

_trainer = BaseTrainer(
    title='RGTransformer',
    area=area,
    uid=trainer_uid,
    model_class=RGTransformer,
    dataset_class=OISSTMonthlyDataset,
    save_path=f'{model_path}/rg_transformer.pkl',
    use_checkpoint=use_checkpoint,
    checkpoint_path=f'out/checkpoints/RGTransformer/last.ckpt',
    dataset_params=dataset_params,
    trainer_params=trainer_params,
    model_params=rg_transformer_m_params,
)

_trainer.train()